# TCM-Classics-Bench · 中医古籍源文本约束型测评基准

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariskang/TCM-Classics-Bench/blob/claude/tcm-classics-bench-dataset-tu71dt/notebooks/TCM_Classics_Bench.ipynb)

**面向中医古籍大模型的源文本约束型测评基准** — *A Source-Grounded Benchmark for
Large Language Models on Classical Chinese Medicine Texts.*

本 Notebook 在 Colab 中**一键复现全部功能**：

1. 克隆仓库、安装依赖
2. 下载并解压「中醫笈成 / Jicheng-TCM」古籍语料（snapshot `book-20180111`，803 部）
3. 解析编目 → `catalog.jsonl`
4. **简易模式**：数秒生成 5000 道已校验测试题（无需 API key）
5. 源文本约束校验（evidence 回指）
6. 可视化分布（任务 / 书目 / 朝代 / 品质）
7. 浏览样题（T1 句读、T6 方剂结构解析）
8. **多 provider LLM 接入**：Anthropic / Azure / Poe / LiteLLM（T2–T12）
9. **并发生成 + 实时写入 Google Drive + 进度条**（断线不丢、线程池并发、tqdm）
10. 全量构建 `build_release` + `STATS.json`
11. 下载结果、运行测试

> 运行环境：菜单 *代码执行程序 → 全部运行*。无需 GPU。整段约 1–2 分钟。
> 数据授权见仓库 `DATA_LICENSE.md`（公共领域文本编辑 CC0；站方自创 CC BY 4.0，逐页核查）。

## 1. 环境准备 · Clone & install

In [ ]:
import os

REPO = "https://github.com/pariskang/TCM-Classics-Bench.git"
BRANCH = "claude/tcm-classics-bench-dataset-tu71dt"   # 合并到 main 后改成 "main"

if not os.path.exists("/content/TCM-Classics-Bench"):
    !git clone --branch $BRANCH --depth 1 $REPO /content/TCM-Classics-Bench
%cd /content/TCM-Classics-Bench
!git log --oneline -1

In [ ]:
# 核心依赖：繁简转换 + 纯 Python 7z 解压 + 进度条
!pip install -q opencc py7zr tqdm
# 可选 LLM provider（用到再装；任选其一/多）：
# !pip install -q anthropic        # provider=anthropic
# !pip install -q openai           # provider=azure / poe
# !pip install -q litellm          # provider=litellm
print("deps ready")

## 2. 下载语料 · Download & extract corpus

从 [中醫笈成](https://jicheng.tw/tcm/download.html) 下载 `book-20180111.7z`（约 69 MB）
并用 `py7zr` 解压到 `corpus_src/book/`（每部古籍一个目录）。

In [ ]:
import os, urllib.request, py7zr

URL = "https://jicheng.tw/files/jcw/book-20180111.7z"
ROOT = "corpus_src/book"
os.makedirs(ROOT, exist_ok=True)
archive = "corpus_src/book-20180111.7z"

if not os.path.exists(archive):
    print("downloading ~69MB ...")
    urllib.request.urlretrieve(URL, archive)

if len(os.listdir(ROOT)) <= 1:
    print("extracting...")
    with py7zr.SevenZipFile(archive, "r") as z:
        z.extractall(ROOT)

print("books extracted:", len(os.listdir(ROOT)))

## 3. 解析与编目 · Catalog（全部 803 部书的元数据）

In [ ]:
!python -m tcm_bench catalog --root corpus_src/book --out data/catalog.jsonl

import pandas as pd, json
cat = pd.DataFrame(json.loads(l) for l in open("data/catalog.jsonl"))
print("books:", len(cat))
display(cat[["book_title_trad","author","dynasty","category_level_1",
             "quality_score_source","quality_tier"]].head(10))
print("\n按一级分类:"); display(cat["category_level_1"].value_counts())
print("\n按品质分层:"); display(cat["quality_tier"].value_counts())

## 4. 简易模式 · 5000 道测试题（确定性，无需 API key）

`simple` 一条命令：摄取 pilot 语料 → 生成 T1（句读恢复）+ T6（方剂结构解析）→
**逐题源文本校验** → 按 (任务, 书目) 轮转均衡抽样到 N 道。

In [ ]:
!python -m tcm_bench simple --root corpus_src/book --n 5000 --out test_5k.jsonl

In [ ]:
import pandas as pd, json
items = pd.DataFrame(json.loads(l) for l in open("test_5k.jsonl"))
print("total:", len(items))
print("\n按任务:"); display(items["task_code"].value_counts())
print("\n按书目:"); display(items["book_id"].value_counts())

## 5. 可视化 · 分布图

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
items["task_code"].value_counts().plot.bar(ax=ax[0], title="Items by task", rot=0)
items["book_id"].value_counts().plot.barh(ax=ax[1], title="Items by book")
ax[1].set_yticklabels([])  # CJK 字体在 Colab 默认缺失，标签见上方表格
plt.tight_layout(); plt.show()

## 6. 浏览样题 · Inspect items（T1 句读 / T6 方剂解析）

In [ ]:
import json, textwrap

def show(it):
    print("="*70)
    print("task :", it["task_code"], "|", it["task"], "| difficulty:", it.get("difficulty"))
    print("book :", it["evidence"]["book_title_trad"], "| 底本:", it["evidence"].get("base_text"))
    print("Q    :", it["question"])
    print("context:", textwrap.shorten(str(it["context"]), 160))
    print("answer :", json.dumps(it["answer"], ensure_ascii=False)[:400])
    if it.get("safety_note"): print("safety :", it["safety_note"])

rows = [json.loads(l) for l in open("test_5k.jsonl")]
show(next(r for r in rows if r["task_code"] == "T1"))
show(next(r for r in rows if r["task_code"] == "T6"))

## 7. 源文本约束校验 · Validation

每道题都必须能回指原文：evidence span、方名、药材、剂量都要在原文出现；
`external_required` 题被逐出正式集。`simple` 已内置校验，这里对仓库内置的
`data/bench/` 样本再独立校验一次（对应 `data/corpus/`）。

In [ ]:
!python -m tcm_bench validate --items data/bench/pilot2_zhongjing.jsonl --corpus data/corpus/pilot2_zhongjing.jsonl

## 8. 多 provider LLM 接入 · T2–T12（可选）

确定性生成器只覆盖 T1 / T6。其余任务（翻译、术语、实体、方证、安全等）走 LLM，
统一接口支持四家 provider。**填入对应密钥后**再运行本节。

In [ ]:
#@title 选择 provider 并填写密钥 { display-mode: "form" }
import getpass, os
PROVIDER = "anthropic" #@param ["anthropic", "azure", "poe", "litellm"]
MODEL = "" #@param {type:"string"}
MODEL = MODEL.strip() or None

def setkey(name):
    if not os.environ.get(name):
        os.environ[name] = getpass.getpass(f"{name}: ")

if PROVIDER == "anthropic":
    !pip install -q anthropic
    setkey("ANTHROPIC_API_KEY")
elif PROVIDER == "poe":
    !pip install -q openai
    setkey("POE_API_KEY")
elif PROVIDER == "azure":
    !pip install -q openai
    for k in ["AZURE_OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT"]:
        setkey(k)
    os.environ.setdefault("AZURE_OPENAI_API_VERSION", "2024-06-01")
elif PROVIDER == "litellm":
    !pip install -q litellm
print("provider:", PROVIDER, "| model:", MODEL)

In [ ]:
# 取几条 pilot 段落，跑 LLM 任务（T2 文白翻译 + T6）做演示
import json
from pathlib import Path
from tcm_bench import ingest
from tcm_bench.generate import LLMGenerator, generate_items
from tcm_bench.llm import make_client
from tcm_bench.validate import validate_item

recs = [r.to_dict() for r in ingest.ingest_book(Path("corpus_src/book/難經"), "2026-06-26")][:3]
gen = LLMGenerator(make_client(PROVIDER, MODEL))
for it in generate_items(recs, ["T2"], llm=gen):
    d = it.to_dict()
    ok = validate_item(d, next(r["raw_text_trad"] for r in recs if r["passage_id"] == d["passage_id"])).ok
    print("[", "OK" if ok else "FAIL", "]", d["task"], "->",
          json.dumps(d["answer"], ensure_ascii=False)[:200])

你也可以直接用命令行（任选 provider；`--workers` 并发、`--progress` 进度条）：

```bash
# Poe，16 路并发 + 进度条
export POE_API_KEY=...
python -m tcm_bench simple --root corpus_src/book --n 300 --tasks T1 T6 T2 T8 \
       --llm --provider poe --model Claude-Sonnet-4 --workers 16 --progress

# Azure OpenAI
export AZURE_OPENAI_API_KEY=...  AZURE_OPENAI_ENDPOINT=https://<res>.openai.azure.com
python -m tcm_bench generate --corpus data/corpus/pilot1_classics.jsonl \
       --out llm_items.jsonl --tasks T2 T3 --llm --provider azure --model <deployment> \
       --workers 8 --progress

# LiteLLM（通用路由）
python -m tcm_bench generate --corpus data/corpus/pilot1_classics.jsonl \
       --out llm_items.jsonl --tasks T2 --llm --provider litellm \
       --model gemini/gemini-1.5-pro --workers 8 --progress
```

## 9. 并发生成 + 实时写入 Google Drive + 进度条

把生成结果**逐题实时写入 Google Drive**（即使 Colab 断线也不丢），用线程池**并发**
调用 LLM，并用 `tqdm` 显示**进度条**。先挂载 Drive：

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/tcm_classics_bench"
os.makedirs(OUT_DIR, exist_ok=True)
print("output dir:", OUT_DIR)

In [ ]:
#@title 并发生成 + 实时写入 Drive + 进度条 { display-mode: "form" }
N = 5000          #@param {type:"integer"}
TASKS = "T1 T6"   #@param {type:"string"}
WORKERS = 8       #@param {type:"integer"}
USE_LLM = False   #@param {type:"boolean"}

import os, json, random
from pathlib import Path
from tqdm.auto import tqdm
from tcm_bench import ingest, taxonomy
from tcm_bench.generate import generate_items, generate_items_concurrent, LLMGenerator
from tcm_bench.llm import make_client
from tcm_bench.validate import validate_item

# 1) 摄取 pilot 语料并打散（确定性 seed，保证可复现 + 均衡）
records = []
for pilot in taxonomy.PILOT_CORPORA:
    for name in taxonomy.PILOT_CORPORA[pilot]:
        d = Path("corpus_src/book") / name
        if (d / "index.txt").exists():
            records += [r.to_dict() for r in ingest.ingest_book(d, "2026-06-26", min_chars=20)]
random.Random(20260626).shuffle(records)
src = {r["passage_id"]: r["raw_text_trad"] for r in records}

# 2) 选择 LLM（USE_LLM=True 时；provider/model 取自上一节表单，缺省 anthropic）
llm = None
if USE_LLM:
    prov = globals().get("PROVIDER", "anthropic")
    mdl = globals().get("MODEL", None)
    llm = LLMGenerator(make_client(prov, mdl))
    gen = generate_items_concurrent(records, TASKS.split(), llm=llm, max_workers=WORKERS)
else:
    gen = generate_items(records, TASKS.split())   # 确定性，飞快

# 3) 实时写入 Drive + 进度条 + 逐题源文本校验
out_path = os.path.join(OUT_DIR, "tcm_questions.jsonl")
n_ok = n_bad = 0
with open(out_path, "w", encoding="utf-8") as fh, tqdm(total=N, unit="题", desc="生成") as bar:
    for it in gen:
        d = it.to_dict()
        if validate_item(d, src.get(d["passage_id"], "")).ok:
            fh.write(json.dumps(d, ensure_ascii=False) + "\n"); fh.flush()  # 实时落盘
            n_ok += 1; bar.update(1)
        else:
            n_bad += 1
        if n_ok >= N:
            break
print(f"\n完成：{n_ok} 题已实时写入 {out_path}（校验失败 {n_bad}）")

## 10. 全量构建 · build_release（catalog + 三个 pilot + bench + STATS）

In [ ]:
!python scripts/build_release.py --root corpus_src/book
import json
print(json.dumps(json.load(open("data/STATS.json")), ensure_ascii=False, indent=2)[:1500])

## 11. 下载结果 · Download

In [ ]:
from google.colab import files
files.download("test_5k.jsonl")        # 5000 道题
# files.download("data/catalog.jsonl") # 803 部书元数据

## 12. 单元测试 · Tests

In [ ]:
!python -m pytest -q

---

仓库：**pariskang/TCM-Classics-Bench**（分支 `claude/tcm-classics-bench-dataset-tu71dt`）
· 协议见 `PROTOCOL.md` · 授权见 `DATA_LICENSE.md`
· 题目仅作古籍理解测评，**不构成任何用药建议**。